In [1]:
import sympy as sp
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import special
import math
import pandas as pd

def moshinsky_function(x, k, t, m, hbar):
    M = (1/2) * special.erfc(
        (x - (hbar * k * t / m)) /
        (np.sqrt(2 * hbar * 1j * t / m))
    ) * np.exp(
        1j * (k * x - hbar * (k**2) * t / (2 * m))
    )  # Moshinsky function

    return M


# Initial parameters

hbar = 6.582119e-1   # Reduced Planck constant in eV-fs
V0 = 0.1             # Potential barrier in eV
a = 500              # Barrier width in Angstroms
c = 3e8              # Speed of light in m/s
x = 500              # Observation position in Angstroms

# t = 200            # Time in fs

E0 = 0.001           # Incident energy in eV

m = 0.067 * 1e10 * 0.510998e6 / c**2
# Effective electron mass (0.067 m_e)

m_e = 0.067 * 5.6791e-8

K0 = np.sqrt((2 * m * E0) / (hbar**2))
Kv0 = np.sqrt((2 * m * V0) / (hbar**2))

Alpha = a * Kv0
Gamma = math.tanh(Alpha) * (Kv0 / 2)

Tk = (K0**2) / (
    math.cosh(Alpha)**2 *
    (K0**2 + Gamma**2)
)

# Initial values for the magnetic field contribution

omega = 0.23
# Larmor frequency in 1/fs

# Gyromagnetic ratio = 1.7808e-4 1/fs 1/T

B = omega / 1.7808e-4  # Magnetic field in Tesla

E_larmor = hbar * omega / 2

V_positive = V0 - hbar * omega / 2
V_negative = V0 + hbar * omega / 2

Kv0_positive = np.sqrt((2 * m * V_positive) / (hbar**2))
Kv0_negative = np.sqrt((2 * m * V_negative) / (hbar**2))

Alpha_positive = a * Kv0_positive
Alpha_negative = a * Kv0_negative

Gamma_positive = math.tanh(Alpha_positive) * Kv0_positive / 2
Gamma_negative = math.tanh(Alpha_negative) * Kv0_negative / 2

Tk_up = (K0**2) / (
    math.cosh(Alpha_positive)**2 *
    (K0**2 + Gamma_positive**2)
)

Tk_down = (K0**2) / (
    math.cosh(Alpha_negative)**2 *
    (K0**2 + Gamma_negative**2)
)

Tk_total = Tk_up + Tk_down

# Tunneling time according to Rybachenko

kappa = np.sqrt(Kv0**2 - K0**2)

Transversal_Time = (hbar * K0) / (V0 * kappa)

print(Transversal_Time)

# Obtain the maximum value of the function
# and the corresponding time

t_initial = 200     # Initial time
t_final = 600       # Final time
dt = 0.01           # Time step

i = 0
t_maximum = 0

X = np.zeros(int((t_final - t_initial) / dt) + 1)
# Function values at each point

while t_initial <= t_final:

    Psi_up = (
        (1 / np.sqrt(2)) *
        (
            (K0 - 1j * Gamma_positive) /
            (
                math.cosh(Alpha_positive) *
                (K0**2 + Gamma_positive**2)
            )
        )
    ) * (
        K0 * moshinsky_function(
            x - a,
            K0,
            t_initial,
            m,
            hbar
        )
        + 1j * Gamma_positive *
        moshinsky_function(
            x - a,
            -1j * Gamma_positive,
            t_initial,
            m,
            hbar
        )
    )

    Psi_up_density = np.real(
        Psi_up * np.conj(Psi_up)
    )

    Psi_down = (
        (1 / np.sqrt(2)) *
        (
            (K0 - 1j * Gamma_negative) /
            (
                math.cosh(Alpha_negative) *
                (K0**2 + Gamma_negative**2)
            )
        )
    ) * (
        K0 * moshinsky_function(
            x - a,
            K0,
            t_initial,
            m,
            hbar
        )
        + 1j * Gamma_negative *
        moshinsky_function(
            x - a,
            -1j * Gamma_negative,
            t_initial,
            m,
            hbar
        )
    )

    Psi_down_density = np.real(
        Psi_down * np.conj(Psi_down)
    )

    # Expectation value calculation

    S_z = (hbar / 2) * (
        Psi_up_density - Psi_down_density
    )

    # Expectation values

    MV_S_z = 2 * np.real(S_z) / Tk_total

    X[i] = MV_S_z

    t_initial += dt
    i += 1

M = max(X)

# Reset time value for the next cycle

t_initial = 200

# Compute the time at which the function
# reaches its maximum value

while t_initial <= t_final:

    Psi_up = (
        (1 / np.sqrt(2)) *
        (
            (K0 - 1j * Gamma_positive) /
            (
                math.cosh(Alpha_positive) *
                (K0**2 + Gamma_positive**2)
            )
        )
    ) * (
        K0 * moshinsky_function(
            x - a,
            K0,
            t_initial,
            m,
            hbar
        )
        + 1j * Gamma_positive *
        moshinsky_function(
            x - a,
            -1j * Gamma_positive,
            t_initial,
            m,
            hbar
        )
    )

    Psi_up_density = np.real(
        Psi_up * np.conj(Psi_up)
    )

    Psi_down = (
        (1 / np.sqrt(2)) *
        (
            (K0 - 1j * Gamma_negative) /
            (
                math.cosh(Alpha_negative) *
                (K0**2 + Gamma_negative**2)
            )
        )
    ) * (
        K0 * moshinsky_function(
            x - a,
            K0,
            t_initial,
            m,
            hbar
        )
        + 1j * Gamma_negative *
        moshinsky_function(
            x - a,
            -1j * Gamma_negative,
            t_initial,
            m,
            hbar
        )
    )

    Psi_down_density = np.real(
        Psi_down * np.conj(Psi_down)
    )

    # Expectation value calculation

    S_z = (hbar / 2) * (
        Psi_up_density - Psi_down_density
    )

    # Expectation values

    MV_S_z = 2 * np.real(S_z) / Tk_total

    H = MV_S_z

    if H == M:
        t_maximum = t_initial
        break

    t_initial += dt

print(
    t_maximum,
    M
)

# Time at which the density reaches its maximum
# value and the corresponding maximum

0.6615278499536149
200 0.27174648393966316
